Cell 1: Install Dependencies & Setup

In [2]:
!pip install -q transformers accelerate xgboost scikit-learn pandas numpy torch joblib

import os
import pandas as pd
import numpy as np
import torch
import gc
from sklearn.model_selection import train_test_split

# Create directories to save models and results
os.makedirs("./models", exist_ok=True)
os.makedirs("./embeddings", exist_ok=True)

Cell 2: Data Loading & Minimal Preprocessing

Note: This assumes you have a combined CSV. If you are loading multiple files, combine them first. The DataFrame must have text, label, and artifact_type columns.

In [3]:
import os
import pandas as pd

# --- LOAD YOUR 4 DATASETS ---
file_artifact_map = {
    "data-augmentation-code_comments.csv": "code_comment",
    "data-augmentation-issues.csv": "issue",
    "data-augmentation-commit-messages.csv": "commit",
    "data-augmentation-pull-requests.csv": "pull_request"
}

frames = []

# If files are in a specific folder, update the path below (e.g., "./data/filename.csv")
for fname, artifact_type in file_artifact_map.items():
    # Check if file exists before trying to read
    if not os.path.exists(fname):
        print(f"WARNING: {fname} not found. Please ensure it is in the working directory.")
        continue

    # Added sep=';' to handle the semicolon delimiter
    df_temp = pd.read_csv(fname, sep=';')

    # Standardize column names just in case
    if 'classification' not in df_temp.columns:
        # Try to find the label column if named differently
        for col in ['label', 'Classification', 'class']:
            if col in df_temp.columns:
                df_temp.rename(columns={col: 'classification'}, inplace=True)
                break

    df_temp['artifact_type'] = artifact_type
    frames.append(df_temp)
    print(f"Loaded {len(df_temp)} rows from {fname}")

df = pd.concat(frames, ignore_index=True)

# 1. Minimal Preprocessing: Drop NAs and short text (<= 2 words)
df = df.dropna(subset=['text', 'classification'])
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].apply(lambda x: len(x.split()) > 2)]

# 2. Standardize Labels to match the Paper's Schema
label_mapping = {
    'non-satd': 'Not-SATD', 'non_debt': 'Not-SATD',
    'design/code': 'C/D', 'code/design_debt': 'C/D', 'design_debt': 'C/D',
    'code_debt': 'C/D', 'design': 'C/D', 'code': 'C/D', 'code/design': 'C/D',
    'requirement': 'REQ', 'requirement_debt': 'REQ', 'requirements': 'REQ',
    'documentation': 'DOC', 'documentation_debt': 'DOC',
    'test': 'TES', 'test_debt': 'TES',
    'without_classification': 'SATD-uncat' # Will be dropped next
}

df['mapped_label'] = df['classification'].astype(str).str.strip().str.lower().map(label_mapping)

# Drop unmapped/ignored labels (defect, architecture, build, uncat, etc.)
df = df.dropna(subset=['mapped_label'])

print("\nData cleaned. Total rows:", len(df))
print("\nDistribution by Artifact Type:")
print(df.groupby(['artifact_type', 'mapped_label']).size().unstack(fill_value=0))

Loaded 68515 rows from data-augmentation-code_comments.csv
Loaded 28183 rows from data-augmentation-issues.csv
Loaded 6300 rows from data-augmentation-commit-messages.csv
Loaded 6273 rows from data-augmentation-pull-requests.csv

Data cleaned. Total rows: 95704

Distribution by Artifact Type:
mapped_label    C/D   DOC  Not-SATD   REQ   TES
artifact_type                                  
code_comment   2684  2701     46832  2201  2635
commit          487   469      4029   513   522
issue          2163  1944     18678  2134  2019
pull_request    499   500      3718   500   476


Cell 3: Stratified 80/10/10 Split (Before Augmentation)

Crucial: We split before augmenting to prevent data leakage (synthetic text from a train row ending up in the test set).

In [4]:
train_frames, val_frames, test_frames = [], [], []

# Split per artifact type and label to maintain stratification
for (artifact_type, label), group in df.groupby(["artifact_type", "mapped_label"]):
    if len(group) < 10: # Skip tiny groups
        continue

    train_part, temp_part = train_test_split(group, train_size=0.8, random_state=42)
    val_part, test_part = train_test_split(temp_part, train_size=0.5, random_state=42)

    train_frames.append(train_part)
    val_frames.append(val_part)
    test_frames.append(test_part)

train_df = pd.concat(train_frames, ignore_index=True)
val_df = pd.concat(val_frames, ignore_index=True)
test_df = pd.concat(test_frames, ignore_index=True)

print(f"Split complete -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Split complete -> Train: 76556 | Val: 9569 | Test: 9579


Cell 4: Qwen LLM Data Augmentation (Train Set Only)

Implements the leader's instruction to use Qwen and the paper's multi-turn persona prompt.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load Qwen Model for Augmentation
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

aug_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def batch_augment_qwen(texts, artifact_type):
    batch_messages = []
    for txt in texts:
        batch_messages.append([
            {"role": "user", "content": "You are a helpful assistant that rephrases text and makes sentences smooth."},
            {"role": "assistant", "content": "Of course! Feel free to provide the text you'd like me to rephrase, and I'll be happy to assist."},
            {"role": "user", "content": f"I will give you a sample {artifact_type} from GitHub, please rephrase it following the style of a programmer, then give me exactly 1 rephrased answer back cleanly without context introduction. Text: '{txt}'"}
        ])

    prompts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(aug_model.device)

    with torch.no_grad():
        generated_ids = aug_model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.7, pad_token_id=tokenizer.pad_token_id)

    responses = []
    for i in range(len(texts)):
        input_len = inputs.input_ids.shape[1]
        gen_tokens = generated_ids[i][input_len:]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        responses.append(decoded if len(decoded) > 5 else texts[i])
    return responses

balanced_train_frames = []

for artifact_type, group in train_df.groupby("artifact_type"):
    # Target = Largest SATD class in this artifact
    satd_only = group[group["mapped_label"] != "Not-SATD"]
    target = int(satd_only["mapped_label"].value_counts().max())

    artifact_frames = []

    for label, subset in group.groupby("mapped_label"):
        current = len(subset)

        if label == "Not-SATD":
            # 1. Downsample Not-SATD
            sampled = subset.sample(n=target, random_state=42)
            artifact_frames.append(sampled)
        elif current >= target:
            # 2. Majority SATD class - just sample
            sampled = subset.sample(n=target, random_state=42)
            artifact_frames.append(sampled)
        else:
            # 3. Minority SATD class - Augment with Qwen
            needed = target - current
            print(f"Augmenting {artifact_type} | {label}: {current} -> +{needed}")
            artifact_frames.append(subset)

            new_rows = []
            texts = subset["text"].tolist()
            batch_size = 16

            while len(new_rows) < needed:
                batch_texts = [texts[i % len(texts)] for i in range(len(new_rows), len(new_rows) + batch_size)]
                try:
                    results = batch_augment_qwen(batch_texts, artifact_type)
                    for res in results:
                        if len(new_rows) < needed:
                            new_rows.append({"text": res, "mapped_label": label, "artifact_type": artifact_type})
                except Exception as e:
                    print(f"Error: {e}")
                    break

            artifact_frames.append(pd.DataFrame(new_rows))

    balanced_train_frames.append(pd.concat(artifact_frames, ignore_index=True))

balanced_train_df = pd.concat(balanced_train_frames, ignore_index=True)

# Free up GPU memory
del aug_model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("\nBalanced Train Distribution:")
print(balanced_train_df.groupby(['artifact_type', 'mapped_label']).size().unstack(fill_value=0))

Cell 5: Generate LLM Embeddings (Qwen3 & E5-Large)

Generates dense vector representations for train, val, and test sets.

In [6]:
!pip install -q tqdm
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
import torch

balanced_train_df = train_df

def extract_vectors(text_series, model_name, is_e5=False, batch_size=128):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 1. Use SDPA for major speedup, force device to "cuda"
    # 2. eval() mode disables dropout for faster inference
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="cuda",
        attn_implementation="sdpa"
    )
    model.eval()

    embeddings = []
    text_list = text_series.tolist()

    # 3. Add a progress bar so you can see the speed
    for i in tqdm(range(0, len(text_list), batch_size), desc=f"Processing {model_name}"):
        batch = text_list[i:i+batch_size]
        if is_e5:
            batch = [f"query: {t}" for t in batch]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, max_length=128, truncation=True).to("cuda")

        # 4. inference_mode is faster than no_grad
        with torch.inference_mode():
            outputs = model(**inputs)

        # Masked Mean Pooling
        mask = inputs.attention_mask.unsqueeze(-1)
        matrix = outputs.last_hidden_state
        vectors = torch.sum(matrix * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

        # L2 Normalize
        vectors = torch.nn.functional.normalize(vectors, p=2, dim=1)
        embeddings.append(vectors.cpu().numpy())

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return np.vstack(embeddings)

splits = {"train": balanced_train_df, "val": val_df, "test": test_df}

# 1. Qwen3 Embeddings
print("Extracting Qwen3-Embedding-0.6B...")
for name, df_split in splits.items():
    emb = extract_vectors(df_split['text'], "Qwen/Qwen3-Embedding-0.6B")
    np.save(f"./embeddings/qwen3_{name}.npy", emb)

# 2. E5-Large Embeddings
print("\nExtracting intfloat/e5-large-v2...")
for name, df_split in splits.items():
    emb = extract_vectors(df_split['text'], "intfloat/e5-large-v2", is_e5=True)
    np.save(f"./embeddings/e5_{name}.npy", emb)

print("\nEmbeddings saved successfully.")

Extracting Qwen3-Embedding-0.6B...
Loading Qwen/Qwen3-Embedding-0.6B...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Processing intfloat/e5-large-v2: 100%|██████████| 75/75 [00:47<00:00,  1.56it/s]


Loading intfloat/e5-large-v2...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Processing intfloat/e5-large-v2: 100%|██████████| 75/75 [00:48<00:00,  1.56it/s]



Embeddings saved successfully.


Cell 6: Train Two-Step Classifiers & Evaluate

Trains Step 1 (Binary) and Step 2 (Multi-class) separately for each artifact type.

In [7]:
import joblib
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Load embeddings and labels
emb_sets = {
    "qwen3": {s: np.load(f"./embeddings/qwen3_{s}.npy") for s in splits},
    "e5":    {s: np.load(f"./embeddings/e5_{s}.npy") for s in splits}
}

# Prepare DataFrames with binary ID label
for s in splits:
    splits[s] = splits[s].copy()
    splits[s]["id_label"] = splits[s]["mapped_label"].apply(lambda x: "Not-SATD" if x == "Not-SATD" else "SATD")

artifact_types = splits["train"]["artifact_type"].unique()
all_results = []

def train_eval(X_tr, y_tr, X_te, y_te, clf_type, embed_name, art_type, task, label_enc):
    if clf_type == "xgboost":
        clf = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, eval_metric="mlogloss", random_state=42, n_jobs=-1)
    else:
        clf = LogisticRegression(max_iter=2000, C=0.5, random_state=42, n_jobs=-1)

    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_te)

    macro_f1 = f1_score(y_te, preds, average="macro", zero_division=0)

    # Save model
    model_path = f"./models/{task}_{art_type}_{embed_name}_{clf_type}.joblib"
    joblib.dump(clf, model_path)

    return macro_f1

# --- STEP 1: IDENTIFICATION (SATD vs Not-SATD) ---
print("="*50)
print("STEP 1: IDENTIFICATION")
print("="*50)

for art_type in artifact_types:
    tr_mask = splits["train"]["artifact_type"] == art_type
    te_mask = splits["test"]["artifact_type"] == art_type

    # Binary labels 0: Not-SATD, 1: SATD
    y_train = (splits["train"].loc[tr_mask, "id_label"] == "SATD").astype(int).values
    y_test = (splits["test"].loc[te_mask, "id_label"] == "SATD").astype(int).values

    for emb_name, emb_data in emb_sets.items():
        X_train = emb_data["train"][tr_mask.values]
        X_test = emb_data["test"][te_mask.values]

        for clf in ["xgboost", "logreg"]:
            f1 = train_eval(X_train, y_train, X_test, y_test, clf, emb_name, art_type, "ident", None)
            all_results.append({"task":"Identification", "artifact":art_type, "embedding":emb_name, "classifier":clf, "macro_f1":f1})
            print(f"[{art_type}] {emb_name} + {clf} -> F1: {f1:.4f}")

# --- STEP 2: CATEGORIZATION (C/D, REQ, TES, DOC) ---
print("\n" + "="*50)
print("STEP 2: CATEGORIZATION")
print("="*50)

for art_type in artifact_types:
    tr_mask = (splits["train"]["artifact_type"] == art_type) & (splits["train"]["id_label"] == "SATD")
    te_mask = (splits["test"]["artifact_type"] == art_type) & (splits["test"]["id_label"] == "SATD")

    # Multiclass labels
    le = LabelEncoder()
    y_train = le.fit_transform(splits["train"].loc[tr_mask, "mapped_label"])
    y_test = le.transform(splits["test"].loc[te_mask, "mapped_label"])

    for emb_name, emb_data in emb_sets.items():
        X_train = emb_data["train"][tr_mask.values]
        X_test = emb_data["test"][te_mask.values]

        for clf in ["xgboost", "logreg"]:
            f1 = train_eval(X_train, y_train, X_test, y_test, clf, emb_name, art_type, "cat", le)
            all_results.append({"task":"Categorization", "artifact":art_type, "embedding":emb_name, "classifier":clf, "macro_f1":f1})
            print(f"[{art_type}] {emb_name} + {clf} -> F1: {f1:.4f}")

# Save results to CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv("./experiment_metrics_summary.csv", index=False)

STEP 1: IDENTIFICATION
[code_comment] qwen3 + xgboost -> F1: 0.9664
[code_comment] qwen3 + logreg -> F1: 0.9336
[code_comment] e5 + xgboost -> F1: 0.9587
[code_comment] e5 + logreg -> F1: 0.9139
[commit] qwen3 + xgboost -> F1: 0.8917
[commit] qwen3 + logreg -> F1: 0.8357
[commit] e5 + xgboost -> F1: 0.8817
[commit] e5 + logreg -> F1: 0.8374
[issue] qwen3 + xgboost -> F1: 0.8656
[issue] qwen3 + logreg -> F1: 0.8381
[issue] e5 + xgboost -> F1: 0.8632
[issue] e5 + logreg -> F1: 0.8426
[pull_request] qwen3 + xgboost -> F1: 0.8642
[pull_request] qwen3 + logreg -> F1: 0.8180
[pull_request] e5 + xgboost -> F1: 0.8703
[pull_request] e5 + logreg -> F1: 0.8312

STEP 2: CATEGORIZATION
[code_comment] qwen3 + xgboost -> F1: 0.9470
[code_comment] qwen3 + logreg -> F1: 0.8694
[code_comment] e5 + xgboost -> F1: 0.9581
[code_comment] e5 + logreg -> F1: 0.8638
[commit] qwen3 + xgboost -> F1: 0.9639
[commit] qwen3 + logreg -> F1: 0.8996
[commit] e5 + xgboost -> F1: 0.9487
[commit] e5 + logreg -> F1: 0.92

Cell 7: Compare Results Against Baselines

Generates the final comparison table requested by the leader.

In [8]:
# Baselines from the paper / supervisor's replication
baseline_ident = {
    "code_comment": 0.939, "issue": 0.878, "pull_request": 0.862, "commit": 0.910
}
baseline_cat = {
    "code_comment": 0.882, "issue": 0.899, "pull_request": 0.876, "commit": 0.980
}

def build_table(task_name, baseline_dict):
    task_results = results_df[results_df["task"] == task_name].copy()
    rows = []
    for art_type, baseline_f1 in baseline_dict.items():
        rows.append({
            "Artifact": art_type, "Embedding": "Baseline", "Classifier": "Paper/Repl",
            "Macro F1": baseline_f1, "vs Baseline": "— (Baseline)"
        })

        art_results = task_results[task_results["artifact"] == art_type]
        for _, r in art_results.iterrows():
            diff = r["macro_f1"] - baseline_f1
            sign = "+" if diff >= 0 else ""
            rows.append({
                "Artifact": art_type, "Embedding": r["embedding"], "Classifier": r["classifier"],
                "Macro F1": round(r["macro_f1"], 4), "vs Baseline": f"{sign}{diff:.4f}"
            })
    return pd.DataFrame(rows)

print("="*70)
print("IDENTIFICATION TASK COMPARISON")
print("="*70)
print(build_table("Identification", baseline_ident).to_string(index=False))

print("\n" + "="*70)
print("CATEGORIZATION TASK COMPARISON")
print("="*70)
print(build_table("Categorization", baseline_cat).to_string(index=False))

# Overall best embedding
print("\n" + "="*70)
print("BEST EMBEDDING MODEL (Averaged across all artifacts & tasks)")
print("="*70)
avg_perf = results_df.groupby("embedding")["macro_f1"].mean().sort_values(ascending=False)
print(avg_perf.round(4))

avg_combo = results_df.groupby(["embedding", "classifier"])["macro_f1"].mean().sort_values(ascending=False)
print("\nBEST OVERALL COMBINATION:")
print(avg_combo.round(4))

IDENTIFICATION TASK COMPARISON
    Artifact Embedding Classifier  Macro F1  vs Baseline
code_comment  Baseline Paper/Repl    0.9390 — (Baseline)
code_comment     qwen3    xgboost    0.9664      +0.0274
code_comment     qwen3     logreg    0.9336      -0.0054
code_comment        e5    xgboost    0.9587      +0.0197
code_comment        e5     logreg    0.9139      -0.0251
       issue  Baseline Paper/Repl    0.8780 — (Baseline)
       issue     qwen3    xgboost    0.8656      -0.0124
       issue     qwen3     logreg    0.8381      -0.0399
       issue        e5    xgboost    0.8632      -0.0148
       issue        e5     logreg    0.8426      -0.0354
pull_request  Baseline Paper/Repl    0.8620 — (Baseline)
pull_request     qwen3    xgboost    0.8642      +0.0022
pull_request     qwen3     logreg    0.8180      -0.0440
pull_request        e5    xgboost    0.8703      +0.0083
pull_request        e5     logreg    0.8312      -0.0308
      commit  Baseline Paper/Repl    0.9100 — (Baseline)
